# Downgrade quality: synthetic power-law reference case

This notebook isolates the synthetic power-law analysis from the full demo. 
It compares `ud_grade`, `harmonic_ud_grade`, and `smoothing + ud_grade` 
against a direct target-NSIDE reference map generated from the same random realization.

### Why this case is useful

- The synthetic setup gives a known, controlled reference (`m_ref`) at the output NSIDE.
- Because truth is available, scalar error metrics can be interpreted directly (not only relatively).
- It highlights aliasing, smoothing, and band-limit effects without astrophysical-model uncertainty.


In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt


In [ ]:
def _round_sig(x, sig=2):
    if x == 0 or not np.isfinite(x):
        return float(x)
    return float(np.round(x, sig - int(np.floor(np.log10(abs(x)))) - 1))

def rounded_limits(*maps, q=(0.5, 99.5)):
    vals = np.concatenate([m[np.isfinite(m)] for m in maps])
    lo, hi = np.percentile(vals, q)
    vmin = _round_sig(float(lo), sig=2)
    vmax = _round_sig(float(hi), sig=2)
    if vmin == vmax:
        vmax = vmin + 1.0
    return vmin, vmax

def proj_panels(maps, titles, cmap='RdBu_r', q=(0.5, 99.5), ncol=3, xsize=2200, symmetric=False, unit=''):
    vmin, vmax = rounded_limits(*maps, q=q)
    if symmetric:
        lim = max(abs(vmin), abs(vmax))
        vmin, vmax = -lim, lim
    print(f'vmin={vmin}, vmax={vmax}')
    n = len(maps)
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    plt.figure(figsize=(6.8*ncol, 4.8*nrow))
    for i, (m, t) in enumerate(zip(maps, titles), start=1):
        hp.projview(
            m,
            sub=(nrow, ncol, i),
            title=t,
            min=vmin,
            max=vmax,
            cmap=cmap,
            graticule=True,
            xsize=xsize,
            unit=unit,
            cb_orientation='horizontal',
        )

def compare_to_ref(m, m_ref, cl, cl_ref, ell_min=2):
    good = np.isfinite(m) & np.isfinite(m_ref)
    m0 = m[good]
    r0 = m_ref[good]
    rmse_rel = np.sqrt(np.mean((m0-r0)**2)) / np.std(r0)
    mae_rel = np.mean(np.abs(m0-r0)) / np.std(r0)
    corr = np.corrcoef(m0, r0)[0, 1]
    spec_rel_l2 = np.linalg.norm(cl[ell_min:] - cl_ref[ell_min:]) / np.linalg.norm(cl_ref[ell_min:])
    return {
        'rmse_rel_std': float(rmse_rel),
        'mae_rel_std': float(mae_rel),
        'map_corr': float(corr),
        'spec_rel_l2': float(spec_rel_l2),
    }

def moll_diff_panels(diff_maps, titles, unit='', q=(0.2, 99.8)):
    vmin, vmax = rounded_limits(*diff_maps, q=q)
    lim = max(abs(vmin), abs(vmax))
    vmin, vmax = -lim, lim
    print(f'moll diff scale: vmin={vmin}, vmax={vmax}')
    n = len(diff_maps)
    fig = plt.figure(figsize=(6.8*n, 5.0))
    for i, (m, t) in enumerate(zip(diff_maps, titles), start=1):
        hp.mollview(
            m,
            fig=fig.number,
            sub=(1, n, i),
            title=t,
            min=vmin,
            max=vmax,
            cmap='RdBu_r',
            unit=unit,
            cbar=True,
        )


## 1) Synthetic power-law map with reference at target NSIDE

### Method notes

- `ud_grade`: pixel-space averaging/rebinning; fast, but can leak high-$\ell$ power into lower multipoles when downgrading.
- `harmonic_ud_grade` (default `lmax`): harmonic-space route with conservative band-limit choice.
- `harmonic_ud_grade` (`lmax=3*nside_out-1`): uses the full target HEALPix band-limit.
- `smoothing + ud_grade`: pre-smooth then pixel downgrade to reduce aliasing at the cost of extra blur.

The synthetic reference is formed by truncating the same input $a_{\ell m}$ realization at `lmax_out` and synthesizing directly at `nside_out`.


In [ ]:
nside_in = 256
nside_out = 32
lmax_in = 3 * nside_in - 1
lmax_out = 3 * nside_out - 1

ell = np.arange(lmax_in + 1)
cl = np.zeros(lmax_in + 1)
cl[2:] = (ell[2:] / 80.0) ** (-2.7)
cl /= cl[80]

np.random.seed(1234)
alm_in = hp.synalm(cl, lmax=lmax_in)
m_in = hp.alm2map(alm_in, nside=nside_in, lmax=lmax_in, pixwin=False)

# Reference map: directly synthesize at output NSIDE from the same realization
alm_ref = hp.resize_alm(alm_in, lmax_in, lmax_in, lmax_out, lmax_out)
m_ref = hp.alm2map(alm_ref, nside=nside_out, lmax=lmax_out, pixwin=False)

In [ ]:
m_ud = hp.ud_grade(m_in, nside_out=nside_out)
m_harm_default = hp.harmonic_ud_grade(m_in, nside_out=nside_out)
m_harm_lmax_out = hp.harmonic_ud_grade(m_in, nside_out=nside_out, lmax=lmax_out)
m_smooth_ud = hp.ud_grade(hp.smoothing(m_in, fwhm=np.radians(30/60)), nside_out=nside_out)

In [ ]:
proj_panels(
    [m_ref, m_ud, m_harm_default, m_harm_lmax_out, m_smooth_ud],
    ['reference', 'ud_grade', 'harmonic (default)', 'harmonic (lmax=3N-1)', 'smoothing + ud'],
    cmap='RdBu_r',
    q=(0.5, 99.5),
    unit='arb.',
)


### How to read the spectra panel

- Compare each method to the **direct reference** curve over the full multipole range.
- Deviations at higher $\ell$ indicate where downgrade choices are most visible.
- A flatter mismatch trend usually indicates broad smoothing bias; oscillatory mismatch often indicates mode-coupling/aliasing effects.


In [ ]:
cl_ref = hp.anafast(m_ref, lmax=lmax_out)
cl_ud = hp.anafast(m_ud, lmax=lmax_out)
cl_harm_default = hp.anafast(m_harm_default, lmax=lmax_out)
cl_harm_lmax_out = hp.anafast(m_harm_lmax_out, lmax=lmax_out)
cl_smooth_ud = hp.anafast(m_smooth_ud, lmax=lmax_out)
ell_out = np.arange(lmax_out + 1)

plt.figure(figsize=(8,4.5))
plt.loglog(ell_out[2:], cl_ref[2:], label='reference', lw=2)
plt.loglog(ell_out[2:], cl_ud[2:], label='ud_grade')
plt.loglog(ell_out[2:], cl_harm_default[2:], label='harmonic default')
plt.loglog(ell_out[2:], cl_harm_lmax_out[2:], label='harmonic lmax=3N-1')
plt.loglog(ell_out[2:], cl_smooth_ud[2:], label='smoothing + ud')
plt.xlabel(r'$\ell$')
plt.ylabel(r'$C_\ell$')
plt.title('Synthetic spectra vs reference')
plt.grid(alpha=0.25)
plt.legend()


In [ ]:
synthetic_metrics = {
    'ud_grade': compare_to_ref(m_ud, m_ref, cl_ud, cl_ref),
    'harmonic_default': compare_to_ref(m_harm_default, m_ref, cl_harm_default, cl_ref),
    'harmonic_lmax_3N-1': compare_to_ref(m_harm_lmax_out, m_ref, cl_harm_lmax_out, cl_ref),
    'smoothing_plus_ud': compare_to_ref(m_smooth_ud, m_ref, cl_smooth_ud, cl_ref),
}
synthetic_metrics

## Synthetic quality metrics (Matplotlib)
**Scope:** these scalar metrics are computed **only for the synthetic power-law experiment** (Section 1), where a known target-NSIDE truth map exists.


This section summarizes how each downgrade method matches the direct target-NSIDE reference map. All metrics are computed against that reference and shown as a 2x2 matrix of bars.

**Definition of `ref` used below**

- `ref` means the synthetic target-NSIDE map built directly from the same realization, i.e. `m_ref` from `alm_ref`.
- In pixel-space metrics, `ref` is `m_ref` evaluated on the same valid pixels as the test map.
- In spectrum-space metrics, `ref` is `cl_ref = hp.anafast(m_ref, lmax=lmax_out)`.
- In `std(ref)`, the denominator is `np.std(m_ref[good])`, where `good` is the finite-pixel mask used for the method map and `m_ref`.

### What each statistic means

- `RMSE / std(ref)`: $\sqrt{\mathrm{mean}((m - m_{\mathrm{ref}})^2)} / \mathrm{std}(m_{\mathrm{ref}})$ on valid pixels. This emphasizes larger deviations because errors are squared before averaging. Lower is better.
- `MAE / std(ref)`: $\mathrm{mean}(|m - m_{\mathrm{ref}}|) / \mathrm{std}(m_{\mathrm{ref}})$ on valid pixels. This is more robust to outliers than RMSE. Lower is better.
- `relative spectrum L2`: $||C_\ell - C_{\ell, \mathrm{ref}}||_2 / ||C_{\ell, \mathrm{ref}}||_2$ over multipoles used in the notebook (from $\ell_{\mathrm{min}}=2$ to $\ell_{\mathrm{max, out}}$). This measures spectral-shape mismatch in harmonic space. Lower is better.
- `1 - map correlation`: $1 - \mathrm{corr}(m, m_{\mathrm{ref}})$ on valid pixels. A perfect correlation gives zero; larger values indicate morphological disagreement. Lower is better.

These four statistics are complementary: RMSE/MAE quantify pixel-domain amplitude error, `relative spectrum L2` quantifies harmonic-domain mismatch, and `1 - correlation` focuses on pattern agreement independent of overall scaling.

### Practical interpretation

- Use absolute metrics to answer: *"Which method is closest to truth in an absolute sense?"*
- If methods are tied in RMSE/MAE but differ in `spec_rel_l2`, prefer the one with lower spectral mismatch for power-spectrum analyses.


In [ ]:
method_order = ["harmonic_lmax_3N-1", "harmonic_default", "smoothing_plus_ud", "ud_grade"]
method_labels = {
    "harmonic_lmax_3N-1": "harmonic (lmax=3N-1)",
    "harmonic_default": "harmonic (default lmax)",
    "smoothing_plus_ud": "smoothing + ud_grade",
    "ud_grade": "ud_grade",
}
metric_labels = {
    "rmse_rel_std": "RMSE / std(ref)",
    "mae_rel_std": "MAE / std(ref)",
    "spec_rel_l2": "relative spectrum L2",
    "one_minus_corr": "1 - map correlation",
}
metrics_plot = {}
for method, vals in synthetic_metrics.items():
    metrics_plot[method] = {
        "rmse_rel_std": vals["rmse_rel_std"],
        "mae_rel_std": vals["mae_rel_std"],
        "spec_rel_l2": vals["spec_rel_l2"],
        "one_minus_corr": 1.0 - vals["map_corr"],
    }
metric_order = ["rmse_rel_std", "mae_rel_std", "spec_rel_l2", "one_minus_corr"]


### Absolute metrics matrix

The first matrix shows absolute values of each metric for each method on a log y-scale, always relative to `ref` (`m_ref` and `cl_ref` as defined above).

- Interpretation: compare bars within each panel; shorter bars indicate better agreement with `ref`.
- Why log scale: values can differ by orders of magnitude (especially for near-exact harmonic cases).
- Use this matrix to assess raw performance against the synthetic target-NSIDE truth, independent of any chosen baseline method.


In [ ]:
colors = ["#1b9e77", "#d95f02", "#7570b3", "#666666"]
x = np.arange(len(method_order))

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, metric in zip(axes.flat, metric_order):
    values = np.array([metrics_plot[m][metric] for m in method_order], dtype=float)
    bars = ax.bar(x, values, color=colors)
    ax.set_yscale("log")
    ax.set_title(metric_labels[metric])
    ax.set_xticks(x, [method_labels[m] for m in method_order], rotation=20, ha="right")
    ax.set_ylabel("value (log scale)")
    ax.grid(axis="y", alpha=0.3)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2.0, v, f"{v:.2e}", ha="center", va="bottom", fontsize=8)

fig.suptitle("Absolute quality metrics matrix (all lower = better)")
plt.show()


### Residual-map diagnostics

- Residual maps (`method - ref`) expose where errors localize on the sky.
- Large coherent structures suggest systematic transfer-function mismatch.
- Fine-grained speckle at small scales is often consistent with high-`\ell` leakage or aggressive truncation choices.


In [ ]:
proj_panels(
    [m_ud - m_ref, m_harm_default - m_ref, m_harm_lmax_out - m_ref, m_smooth_ud - m_ref],
    ['ud - ref', 'harm(default) - ref', 'harm(lmax=3N-1) - ref', 'smooth+ud - ref'],
    cmap='RdBu_r',
    q=(0.1, 99.9),
    symmetric=True,
)


### Synthetic Mollweide differences vs reference
Same residuals shown as Mollweide maps for direct visual comparison.


In [ ]:
moll_diff_panels(
    [m_ud - m_ref, m_harm_default - m_ref, m_harm_lmax_out - m_ref, m_smooth_ud - m_ref],
    ['ud - ref', 'harm(default) - ref', 'harm(lmax=3N-1) - ref', 'smooth+ud - ref'],
    unit='arb.',
    q=(0.1, 99.9),
)


## Conclusion

Based on this synthetic power-law demonstration, we can draw the following conclusions:

1. **`ud_grade` (pure spatial downsampling):**
    - Introduces the highest errors overall, both visually (large residuals) and spectrally (most significant deviation from the reference $C_\ell$).
    - Often suffers from aliasing/leakage of small-scale power into larger scales since the high-$\ell$ power is not properly band-limited before decimation.

2. **`smoothing + ud_grade`:**
    - Adding a Gaussian smoothing step before spatial decimation substantially mitigates the aliasing effects.
    - Yields much better morphological and spectral agreement than `ud_grade` alone, proving it's an essential stabilization step when strictly using spatial downsampling. 
    - However, choosing the optimal FWHM is not always straightforward, and it doesn't match the true target-NSIDE map as perfectly as harmonic methods.

3. **`harmonic_ud_grade`:**
    - Clearly dominates in overall accuracy. The resulting maps and spectra perfectly (or near-perfectly) track the theoretical reference map.
    - Providing `lmax=3*nside_out-1` explicitly gives the sharpest agreement with the reference truth, ensuring that exact band-limiting math is respected without arbitrary truncation.
    - **Recommendation:** Whenever the computational cost of spherical harmonic transforms is acceptable (e.g., when partial-sky masked artifacts are not a severe issue, or maps are full-sky), **`hp.harmonic_ud_grade` should be the preferred method** to reduce map resolution since it rigorously handles the angular frequency band limits.